# SLM-KDN Colab Pro A100: RAG + Training + Testing

Use this notebook after syncing your repo to `master`. It clones the GitHub repo, installs dependencies, checks the A100, rebuilds the RAG index over `rag-doc/ex3300.pdf`, runs retrieval smoke tests, then runs preprocess, training, inference, and evaluation.

Before running: in Colab, choose **Runtime -> Change runtime type -> GPU**, then pick an A100 if available.

## 1. Check GPU

In [1]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

Mon May 11 18:09:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Clone Your Repo from Master

In [2]:
%cd /content
!rm -rf slm-kdn
!git clone --branch master https://github.com/Surya-Prasad/slm-kdn.git
%cd /content/slm-kdn
!git status --short
!git rev-parse --abbrev-ref HEAD
!git log -1 --oneline

/content
Cloning into 'slm-kdn'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 131 (delta 59), reused 71 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 2.73 MiB | 48.27 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/slm-kdn
master
46d09de (HEAD -> master, origin/master, origin/HEAD) Merge pull request #8 from Surya-Prasad/rag-pipeline


## 3. Install Dependencies

This installs the project requirements. `pypdf` is required for `rag-doc/ex3300.pdf` ingestion.

In [3]:
!pip install -q -r requirements.txt
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets pypdf

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 166.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 57.1 MB/s eta 0:00:00


## 4. Hugging Face Login

Run this if your base model requires gated access. Use a token that has access to `meta-llama/Meta-Llama-3-8B`.

In [4]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 5. Confirm EX3300 Guide Is Present

In [5]:
!find rag-doc -maxdepth 2 -type f -print
!ls -lh rag-doc/ex3300.pdf

rag-doc/ex3300.pdf
-rw-r--r-- 1 root root 3.5M May 11 18:09 rag-doc/ex3300.pdf


## 6. Build/Test RAG Retrieval Only

This does not run training. It rebuilds `results/rag_index.pkl` and prints source file, page, score, and chunk preview.

In [6]:
!python src/rag_query.py --rebuild --query "What are the front panel ports on a Juniper EX3300 switch?"
!python src/rag_query.py --query "How do I interpret ALM SYS MST LEDs on an EX3300?" --top_k 3
!python src/rag_query.py --query "What are the console port details for EX3300?" --top_k 3
!python src/rag_query.py --query "What power supply information does the EX3300 guide provide?" --top_k 3

[RAG] query: What are the front panel ports on a Juniper EX3300 switch?
[RAG] 1. source=ex3300.pdf, page=25, score=0.5485, preview=Front Panel of an EX3300 Switch The front panel of an EX3300 switch consists of the following components: • Network ports: • Depending on the switch model, 24 or 48 10/100/1000BASE-T Gigabit Ethernet 
[RAG] 2. source=ex3300.pdf, page=26, score=0.4284, preview=Figure 2: Front Panel of an EX3300 Switch with 24 Gigabit Ethernet Ports g021217 0 1 2 3 ALM EX3300 SYS MST LCD panel Network ports Chassis status LEDs Enter buttonSFP+ uplink ports Menu button Rear P
[RAG] 3. source=ex3300.pdf, page=5, score=0.4206, preview=EX3300 Management Cable Specifications and Pinouts | 68 Management Cable Specifications | 68 Console Port Connector Pinout Information | 69 USB Port Specifications for an EX Series Switch | 70 RJ-45 M
[RAG] 4. source=ex3300.pdf, page=82, score=0.4132, preview=Unpacking and Mounting the EX3300 Switch IN THIS SECTION Unpacking an EX3300 Switch | 82 P

## 7. Preprocess NIT Dataset

This preserves the original NIT pipeline and creates `data/processed/*.jsonl` files.

In [7]:
!bash scripts/run_preprocess.sh
!find data/processed -maxdepth 1 -type f -print | sort
!head -n 1 data/processed/test.jsonl

README.md: 4.31kB [00:00, 7.48MB/s]
NIT_dataset.json: 273kB [00:00, 189MB/s]
Generating train split: 100% 1000/1000 [00:00<00:00, 119095.46 examples/s]
data/processed/clean_test.jsonl
data/processed/noisy_test.jsonl
data/processed/paraphrased_test.jsonl
data/processed/test_intent_only.jsonl
data/processed/test_intent_with_context.jsonl
data/processed/test.jsonl
data/processed/train_intent_only.jsonl
data/processed/train_intent_with_context.jsonl
data/processed/train.jsonl
data/processed/val_intent_only.jsonl
data/processed/val_intent_with_context.jsonl
data/processed/val.jsonl
{"intent": "How to display the LED status", "context": "show chassis led", "target_command": "show chassis led", "target_json": "{\"action\":\"show\",\"parameters\":{\"prefix\":\"\",\"unit\":0,\"vlan_id\":null,\"vlan_name\":\"\"},\"target\":\"\",\"target_type\":\"unknown\"}", "category": ""}


## 8. Rebuild RAG After NIT Preprocessing

After preprocessing, the RAG index includes both the processed NIT JSONL records and the EX3300 guide.

In [8]:
!python src/rag_query.py --rebuild --query "What are the front panel ports on a Juniper EX3300 switch?" --top_k 5

[RAG] query: What are the front panel ports on a Juniper EX3300 switch?
[RAG] 1. source=ex3300.pdf, page=25, score=0.5126, preview=Front Panel of an EX3300 Switch The front panel of an EX3300 switch consists of the following components: • Network ports: • Depending on the switch model, 24 or 48 10/100/1000BASE-T Gigabit Ethernet 
[RAG] 2. source=ex3300.pdf, page=26, score=0.4191, preview=Figure 2: Front Panel of an EX3300 Switch with 24 Gigabit Ethernet Ports g021217 0 1 2 3 ALM EX3300 SYS MST LCD panel Network ports Chassis status LEDs Enter buttonSFP+ uplink ports Menu button Rear P
[RAG] 3. source=ex3300.pdf, page=5, score=0.4189, preview=EX3300 Management Cable Specifications and Pinouts | 68 Management Cable Specifications | 68 Console Port Connector Pinout Information | 69 USB Port Specifications for an EX Series Switch | 70 RJ-45 M
[RAG] 4. source=ex3300.pdf, page=82, score=0.4135, preview=Unpacking and Mounting the EX3300 Switch IN THIS SECTION Unpacking an EX3300 Switch | 82 P

## 9. Train LoRA on A100

The repo config is already A100-oriented. If Colab runs out of memory, lower `training.max_seq_len` or batch settings in `config.yaml` before running this cell.

In [9]:
!bash scripts/run_train.sh

config.json: 100% 654/654 [00:00<00:00, 2.87MB/s]
tokenizer_config.json: 50.6kB [00:00, 96.3MB/s]
tokenizer.json: 9.09MB [00:00, 10.9MB/s]
special_tokens_map.json: 100% 73.0/73.0 [00:00<00:00, 404kB/s]
model.safetensors.index.json: 23.9kB [00:00, 50.4MB/s]
Fetching 4 files: 100% 4/4 [00:42<00:00, 10.59s/it]
Download complete: 100% 16.1G/16.1G [00:42<00:00, 379MB/s]
Loading weights: 100% 291/291 [00:04<00:00, 65.14it/s]
generation_config.json: 100% 177/177 [00:00<00:00, 996kB/s]
Map: 100% 722/722 [00:00<00:00, 870.39 examples/s]
{'loss': '0.4872', 'grad_norm': '1.461', 'learning_rate': '0.000187', 'epoch': '0.2216'}
{'loss': '0.05276', 'grad_norm': '1.422', 'learning_rate': '0.0001725', 'epoch': '0.4432'}
{'loss': '0.01862', 'grad_norm': '0.482', 'learning_rate': '0.000158', 'epoch': '0.6648'}
{'loss': '0.02264', 'grad_norm': '0.3383', 'learning_rate': '0.0001435', 'epoch': '0.8864'}
{'loss': '0.01317', 'grad_norm': '0.4755', 'learning_rate': '0.000129', 'epoch': '1.089'}
{'loss': '0.01

## 10. RAG Inference Smoke Test

Create a tiny input file with EX3300-specific questions and run inference with retrieved EX3300 context. The `--rag_debug` output should show chunks from `ex3300.pdf`.

In [12]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 106.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [15]:
import json, os
os.makedirs("data/processed", exist_ok=True)
rows = [
    {"intent": "What are the front panel ports on a Juniper EX3300 switch?", "context": "", "target_command": "", "category": "rag_ex3300"},
    {"intent": "How do I interpret ALM SYS MST LEDs on an EX3300?", "context": "", "target_command": "", "category": "rag_ex3300"},
    {"intent": "What are the console port details for EX3300?", "context": "", "target_command": "", "category": "rag_ex3300"},
]
with open("data/processed/rag_smoke.jsonl", "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

!python src/infer.py \
  --input_file data/processed/rag_smoke.jsonl \
  --output_file results/predictions/rag_smoke_predictions.jsonl \
  --use_rag \
  --rebuild_rag \
  --rag_debug

!cat results/predictions/rag_smoke_predictions.jsonl

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [00:05<00:00, 57.68it/s]

[INFO] Starting BATCHED inference on 3 instances for rag_smoke.jsonl...
Batches:   0% 0/1 [00:00<?, ?it/s][RAG] query: What are the front panel ports on a Juniper EX3300 switch?
[RAG] 1. source=ex3300.pdf, page=25, score=0.5126, preview=Front Panel of an EX3300 Switch The front panel of an EX3300 switch consists of the following components: • Network ports: • Depending on the switch model, 24 or 48 10/100/1000BASE-T Gigabit Ethernet 
[RAG] 2. source=ex3300.pdf, page=26, score=0.4191, preview=Figure 2: Front Panel of an EX3300 Switch with 24 Gigabit Ethernet Ports g021217 0 1 2 3 ALM EX3300 SYS MST LCD panel Network ports Chassis status LEDs Enter buttonSFP+ uplink ports Menu button Rear P
[RAG] 3. source=ex3300.pdf, page=5, score=0.4189, previe

## 11. Full Test Inference + Evaluation

Use this for the original NIT evaluation. Keep `--use_rag` on if you want retrieved context included during inference.

In [16]:
!mkdir -p results/predictions results/metrics
!python src/infer.py \
  --input_file data/processed/test.jsonl \
  --output_file results/predictions/rag_predictions.jsonl \
  --use_rag \
  --rag_debug

!python src/evaluate.py --pred_file results/predictions/rag_predictions.jsonl
!cat results/metrics/eval_metrics.json

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [00:04<00:00, 58.84it/s]

[INFO] Starting BATCHED inference on 150 instances for test.jsonl...
Batches:   0% 0/5 [00:00<?, ?it/s][RAG] query: How to display the LED status
[RAG] 1. source=ex3300.pdf, page=34, score=0.8327, preview=Table 7: Chassis Status LEDs in an EX3300 Switch State and DescriptionColorLED Label There is no alarm or the switch is halted.UnlitALM (Alarm) There is a major alarm. NOTE: When you connect power to 
[RAG] 2. source=test.jsonl, score=0.6102, preview=Intent: How to display the LED status Context: show chassis led Target command: show chassis led
[RAG] 3. source=ex3300.pdf, page=33, score=0.4141, preview=Table 6: LCD Panel Menu Options (continued) DescriptionMenu If you do not want users to use Maintenance menu options, disable the entire menu 

## 12. Save Results to Google Drive

In [17]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/slm-kdn-results
!cp -r results /content/drive/MyDrive/slm-kdn-results/results-rag-$(date +%Y%m%d-%H%M%S)
!ls -lh /content/drive/MyDrive/slm-kdn-results

Mounted at /content/drive
total 4.0K
drwx------ 5 root root 4.0K May 11 18:43 results-rag-20260511-184340
